<a href="https://colab.research.google.com/github/yh-github/caption_reconstruction/blob/main/notebooks/Video_emb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load data

In [ ]:
# Authenticate
from google.colab import auth
from pathlib import Path
auth.authenticate_user(project_id='gen-lang-client-0839056171')

In [ ]:
# Install necessary libraries
!pip install opencv-python-headless timm
#torch scikit-learn


In [ ]:
# ==============================================================================
# GCS BUCKET SETUP CELL
# ==============================================================================

# 1. Install gcsfuse
# Add the GCS FUSE repository
!echo "deb https://packages.cloud.google.com/apt gcsfuse-`lsb_release -c -s` main" | tee /etc/apt/sources.list.d/gcsfuse.list
# Import the repository's public key
!curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | apt-key add -
# Update the package list and install GCS FUSE
!apt-get update
!apt-get install -y gcsfuse


deb https://packages.cloud.google.com/apt gcsfuse-jammy main
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1022  100  1022    0     0  12785      0 --:--:-- --:--:-- --:--:-- 12936
OK
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [346 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,939 kB]
Get:6 https://packages.cloud.google.com/apt gcsfuse-jammy InRelease [1,227 B]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555

In [ ]:
# 3. Mount the bucket
# Replace 'your-bucket-name-here' with the name of your bucket
BUCKET_NAME = 'yh-ai'
# Create a directory for the mount point
!mkdir -p /mnt/gcs
# Mount the bucket
!gcsfuse --implicit-dirs {BUCKET_NAME} /mnt/gcs

print(f"✅ Bucket '{BUCKET_NAME}' is mounted at /mnt/gcs")

# 4. (Optional) List the contents to verify
print("\nBucket contents:")
!ls -l /mnt/gcs/

{"timestamp":{"seconds":1756065853,"nanos":429713846},"severity":"INFO","message":"Start gcsfuse/3.2.0 (Go version go1.24.5) for app \"\" using mount point: /mnt/gcs\n"}
{"timestamp":{"seconds":1756065853,"nanos":429743884},"severity":"INFO","message":"GCSFuse config","config":{"AppName":"","CacheDir":"","Debug":{"ExitOnInvariantViolation":false,"Fuse":false,"Gcs":false,"LogMutex":false},"DisableAutoconfig":false,"EnableAtomicRenameObject":true,"EnableGoogleLibAuth":false,"EnableHns":true,"EnableNewReader":true,"FileCache":{"CacheFileForRangeRead":false,"DownloadChunkSizeMb":200,"EnableCrc":false,"EnableODirect":false,"EnableParallelDownloads":false,"ExperimentalExcludeRegex":"","ExperimentalParallelDownloadsDefaultOn":true,"MaxParallelDownloads":24,"MaxSizeMb":-1,"ParallelDownloadsPerFile":16,"WriteBufferSize":4194304},"FileSystem":{"DirMode":"755","DisableParallelDirops":false,"ExperimentalEnableDentryCache":false,"ExperimentalEnableReaddirplus":false,"FileMode":"644","FuseOptions":[

In [ ]:
!ls -l /mnt/gcs/wild_videos/*.mp4 | wc -l

100


In [ ]:
root_dir = Path('/mnt/gcs/')

# Process (local GPU)

### --- 2. Video Processing Functions ---

In [ ]:
def bump_path(name_to_bump: str, paths: list[Path]) -> list[Path]:
    for i, path in enumerate(paths):
        if path.name == name_to_bump:
            print(f'Bumping {path.name}')
            paths.insert(0, paths.pop(i))
            return paths
    print('Not bumping anything!')
    return paths

def do_bump(paths: list[Path]) -> list[Path]:
    return bump_path('Survival-Instinct_7-clip-8.mp4', paths)


In [ ]:
import cv2
import torch
import timm
from PIL import Image
from torchvision import transforms
import numpy as np
from pathlib import Path
from collections import defaultdict
import math
import yaml

class VideoEmbedder:
    """
    A class to process video files and generate clip-based embeddings.
    """
    def __init__(self, model_name: str = 'vit_small_patch16_224'):
        """
        Initializes the VideoEmbedder, loading the model and setting up the device.
        """
        self.model_name = model_name
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {self.device}")

        # Load the pre-trained model and move it to the appropriate device
        self.model: torch.nn.Module = timm.create_model(self.model_name, pretrained=True)
        self.model.to(self.device)
        self.model.eval()

        # Get the necessary image transformations for the model
        data_config = timm.data.resolve_model_data_config(self.model)
        self.transform: transforms.Compose = timm.data.create_transform(**data_config, is_training=False)

    def _extract_timestamped_frames(self, video_path: Path, fps: int) -> list[tuple[float, Image.Image]]:
        """
        Extracts frames from a video file at a given rate, returning them with their
        actual timestamps.
        """
        timestamped_frames: list[tuple[float, Image.Image]] = []
        vidcap = cv2.VideoCapture(str(video_path))
        if not vidcap.isOpened():
            print(f"Error: Could not open video file {video_path.name}")
            return timestamped_frames

        video_fps: float = vidcap.get(cv2.CAP_PROP_FPS)
        if video_fps == 0:
            print(f"Warning: Could not get FPS for {video_path.name}. Assuming 30.")
            video_fps = 30

        frame_interval = video_fps / fps
        current_frame_pos = 0.0

        while True:
            vidcap.set(cv2.CAP_PROP_POS_FRAMES, int(current_frame_pos))
            success, image = vidcap.read()
            if not success:
                break

            # Get the actual timestamp from the video capture, which is more accurate
            timestamp_msec = vidcap.get(cv2.CAP_PROP_POS_MSEC)
            timestamp_sec = timestamp_msec / 1000.0

            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            timestamped_frames.append((timestamp_sec, Image.fromarray(image)))

            current_frame_pos += frame_interval

        vidcap.release()
        print(f"Extracted {len(timestamped_frames)} frames from {video_path.name} at {fps} FPS.")
        return timestamped_frames

    def _get_frame_embeddings(self, frames: list[Image.Image]) -> list[np.ndarray]:
        """
        Generates a list of vector embeddings, one for each frame.
        (Private method for internal use)
        """
        if not frames:
            return []

        embeddings: list[np.ndarray] = []
        with torch.no_grad():
            for frame in frames:
                img_tensor: torch.Tensor = self.transform(frame).unsqueeze(0).to(self.device)
                embedding: torch.Tensor = self.model.forward_features(img_tensor)
                embeddings.append(embedding[:, 0].cpu().numpy().flatten())

        return embeddings

    def _group_and_average_embeddings(self, timestamps: list[float], embeddings: list[np.ndarray], clip_size: int) -> np.ndarray:
        """
        Groups embeddings by clip and averages them to get one vector per clip.
        (Private method for internal use)
        """
        if not embeddings:
            return np.array([])

        grouped_embeddings = defaultdict(list)
        for timestamp, embedding in zip(timestamps, embeddings):
            clip_index = int(timestamp // clip_size)
            grouped_embeddings[clip_index].append(embedding)

        averaged_embeddings = [np.mean(grouped_embeddings[key], axis=0) for key in sorted(grouped_embeddings.keys())]

        return np.array(averaged_embeddings)

    def process_directory(self, video_dir: Path, output_dir: Path, fps: int, clip_size: int = 1):
        """
        Finds MP4s, generates embeddings, and saves one averaged vector per clip_size.
        """
        if not video_dir.exists():
            print(f"Error: Video directory not found at '{video_dir}'")
            return

        output_dir.mkdir(parents=True, exist_ok=True)

        with open(output_dir/"metadata.yaml",  'w') as f:
            yaml.dump({
                "type": "video_embeddings",
                "model_name": self.model_name,
                "fps": fps,
                "clip_size": clip_size,
                "input": video_dir.name
            }, f, default_flow_style=False, sort_keys=False)



        video_files = list(video_dir.glob("*.mp4"))
        print(f"Found {len(video_files)} videos to process.")

        for video_path in video_files:
            print(f"\n--- Processing: {video_path.name} ---")

            timestamped_frames = self._extract_timestamped_frames(video_path, fps=fps)
            if not timestamped_frames:
                print(f"Skipping video {video_path.name} as no frames were extracted.")
                continue

            timestamps, frames = zip(*timestamped_frames)

            frame_embeddings = self._get_frame_embeddings(list(frames))
            if not frame_embeddings:
                print(f"Skipping video {video_path.name} as no embeddings were generated.")
                continue

            # Optimization: If fps and clip_size are both 1, we can use the frame embeddings directly.
            if fps == 1 and clip_size == 1:
                clip_embeddings = np.array(frame_embeddings)
            else:
                clip_embeddings = self._group_and_average_embeddings(list(timestamps), frame_embeddings, clip_size)

            # --- VALIDATION STEP ---
            # Base the expected number of clips on the timestamp of the LAST extracted frame.
            last_timestamp = timestamps[-1]
            expected_clips = math.ceil(last_timestamp / clip_size)

            # We allow a small tolerance (e.g., 1) for rounding issues at the very end of the video.
            if abs(clip_embeddings.shape[0] - expected_clips) > 1:
                # Downgraded to a warning instead of an exception
                print(
                    f"WARNING: Discrepancy in '{video_path.name}': "
                    f"Expected ~{expected_clips} vectors based on extracted frames, but generated {clip_embeddings.shape[0]}. "
                    "This is likely due to video encoding/timestamp irregularities."
                )

            output_filepath = output_dir / f"{video_path.stem}.npy"
            np.save(output_filepath, clip_embeddings)

            print(f"Saved {clip_embeddings.shape[0]} vectors to {output_filepath} with shape {clip_embeddings.shape}")


### --- 3. Main Execution ---

In [ ]:
    embedder = VideoEmbedder()

    embedder.process_directory(
        video_dir=root_dir/"wild_videos",
        output_dir=root_dir/"wild_videos_embs",
        fps=1,
        clip_size=1
    )


Using device: cuda
Found 100 videos to process.

--- Processing: 4k-Relaxation_12-clip-6.mp4 ---
Extracted 190 frames from 4k-Relaxation_12-clip-6.mp4 at 1 FPS.
Saved 190 vectors to /mnt/gcs/wild_videos_embs/4k-Relaxation_12-clip-6.npy with shape (190, 384)

--- Processing: 4k-Relaxation_3-clip-4.mp4 ---
Extracted 118 frames from 4k-Relaxation_3-clip-4.mp4 at 1 FPS.
Saved 118 vectors to /mnt/gcs/wild_videos_embs/4k-Relaxation_3-clip-4.npy with shape (118, 384)

--- Processing: AiirSource-Military_1-clip-0.mp4 ---
Extracted 70 frames from AiirSource-Military_1-clip-0.mp4 at 1 FPS.
Saved 70 vectors to /mnt/gcs/wild_videos_embs/AiirSource-Military_1-clip-0.npy with shape (70, 384)

--- Processing: AiirSource-Military_12-manual.mp4 ---
Extracted 66 frames from AiirSource-Military_12-manual.mp4 at 1 FPS.
Saved 66 vectors to /mnt/gcs/wild_videos_embs/AiirSource-Military_12-manual.npy with shape (66, 384)

--- Processing: AiirSource-Military_7-clip-1.mp4 ---
Extracted 65 frames from AiirSourc

In [ ]:
!ls {root_dir/"wild_videos_embs"} | wc -l

100
